
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 08: Métodos de Regresión

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Construir y evaluar modelos de regresión para predecir valores continuos, usando el dataset **Diabetes** de scikit-learn.

## 🗺️ Tabla de Contenido
1. [Introducción](#intro)
2. [El dataset: load_diabetes](#dataset)
3. [train_test_split](#split)
4. [Regresión Lineal simple y múltiple](#lineal)
5. [Regresión Polinómica](#polinomica)
6. [Regularización: Ridge y Lasso](#regularizacion)
7. [Métricas de regresión](#metricas)
8. [Ejemplos de aplicación real](#aplicaciones)
9. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Un modelo de **regresión** predice un **número** (no una categoría): el precio de una casa, la temperatura de mañana, o —como en el caso de hoy— qué tan avanzada estará una enfermedad. La idea central es encontrar una función que, dado un conjunto de variables de entrada, se acerque lo más posible al valor real observado.

<a id="dataset"></a>
## 2. El Dataset: load_diabetes

`sklearn.datasets.load_diabetes()` trae información de 442 pacientes con diabetes: 10 variables (edad, sexo, índice de masa corporal, presión arterial y 6 mediciones de sangre, ya estandarizadas) y una variable objetivo que mide el **avance de la enfermedad un año después** del examen inicial.

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd
import numpy as np

datos = load_diabetes(as_frame=True)
df = datos.frame
print(datos.DESCR[:600])
df.head()

In [ ]:
df.describe()

<a id="split"></a>
## 3. train_test_split

### 🔬 Teoría técnica
Nunca se evalúa un modelo con los mismos datos que se usaron para entrenarlo — eso daría una idea falsamente optimista de qué tan bueno es. Se separa un porcentaje (típicamente 20-30%) como conjunto de **prueba**, que el modelo nunca ve durante el entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Entrenamiento:", X_train.shape, "Prueba:", X_test.shape)

<a id="lineal"></a>
## 4. Regresión Lineal Simple y Múltiple

### 🔬 Teoría técnica
La Regresión Lineal ajusta la mejor línea (o hiperplano, con varias variables) que minimiza la suma de los errores al cuadrado entre la predicción y el valor real:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$$

No tiene hiperparámetros de regularización; los coeficientes se calculan directamente (mínimos cuadrados).

In [ ]:
from sklearn.linear_model import LinearRegression

# Simple: una sola variable (bmi = índice de masa corporal)
modelo_simple = LinearRegression()
modelo_simple.fit(X_train[["bmi"]], y_train)
print("Coeficiente (bmi):", modelo_simple.coef_[0])
print("Intercepto:", modelo_simple.intercept_)

In [ ]:
# Múltiple: todas las variables
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train, y_train)

coeficientes = pd.Series(modelo_multiple.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(coeficientes)

### 💪 Fortalezas y debilidades
- **Fortaleza:** muy interpretable (cada coeficiente dice el efecto de esa variable) y rápido de entrenar.
- **Debilidad:** solo captura relaciones **lineales**; si la relación real es curva, el modelo se queda corto (underfitting).

### 🧠 Resumen para dummies
Cada coeficiente te dice: "si esta variable sube 1 unidad (y las demás no cambian), la predicción sube/baja tanto".

<a id="polinomica"></a>
## 5. Regresión Polinómica

### 🔬 Teoría técnica
Cuando la relación no es una línea recta, se pueden crear nuevas variables elevando las originales a potencias (`x²`, `x³`, ...) y seguir usando Regresión Lineal sobre esas variables transformadas — sigue siendo "lineal" en los coeficientes, pero captura curvas.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

modelo_poli = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
)
modelo_poli.fit(X_train[["bmi"]], y_train)
print("El modelo polinómico ahora usa bmi y bmi² como variables.")

### 💪 Fortalezas y debilidades
- **Fortaleza:** captura relaciones curvas sin cambiar de algoritmo.
- **Debilidad:** con un grado (`degree`) muy alto, el modelo puede sobreajustarse (overfitting) memorizando el ruido de los datos de entrenamiento.

### 🧠 Resumen para dummies
Úsala cuando, al graficar `X` vs `y`, veas claramente una curva y no una línea recta.

<a id="regularizacion"></a>
## 6. Regularización: Ridge (L2) y Lasso (L1)

### 🔬 Teoría técnica
Ridge y Lasso agregan una **penalización** al tamaño de los coeficientes, controlada por el hiperparámetro `alpha`:
- **Ridge (L2):** reduce los coeficientes acercándolos a cero, pero rara vez los deja exactamente en cero. Ayuda con la multicolinealidad.
- **Lasso (L1):** puede llevar coeficientes **exactamente a cero**, eliminando variables — actúa como un método de selección de variables automático.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

lasso = Lasso(alpha=0.5)
lasso.fit(X_train, y_train)

comparacion = pd.DataFrame({
    "Lineal": modelo_multiple.coef_,
    "Ridge (alpha=1.0)": ridge.coef_,
    "Lasso (alpha=0.5)": lasso.coef_,
}, index=X.columns)
comparacion

### 💪 Fortalezas y debilidades
- **Ridge:** bueno cuando todas las variables aportan algo, solo quieres controlar coeficientes muy grandes.
- **Lasso:** bueno cuando sospechas que varias variables son irrelevantes y quieres que el propio modelo las descarte.

### 🧠 Resumen para dummies
Si ves en la tabla anterior que Lasso puso algún coeficiente en `0.0`, esa variable "no aportó" según el modelo con ese `alpha`.

<a id="metricas"></a>
## 7. Métricas de Regresión

### 🔬 Teoría técnica
| Métrica | Qué mide | Sensible a outliers |
|---|---|---|
| **MAE** | Promedio del error absoluto | No |
| **MSE** | Promedio del error al cuadrado | Sí (penaliza fuerte errores grandes) |
| **RMSE** | Raíz del MSE (misma unidad que `y`) | Sí |
| **R²** | % de la varianza explicada (1 = perfecto) | — |

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(modelo, X_eval, y_eval, nombre):
    pred = modelo.predict(X_eval)
    mae = mean_absolute_error(y_eval, pred)
    mse = mean_squared_error(y_eval, pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_eval, pred)
    return {"modelo": nombre, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}


resultados = pd.DataFrame([
    evaluar(modelo_multiple, X_test, y_test, "Lineal múltiple"),
    evaluar(ridge, X_test, y_test, "Ridge"),
    evaluar(lasso, X_test, y_test, "Lasso"),
])
resultados

### 🧠 Resumen para dummies
MAE es "en promedio, cuánto me equivoco". R² es "qué porcentaje de la variabilidad de la enfermedad logro explicar" (más cercano a 1 es mejor; puede ser negativo si el modelo es peor que predecir siempre el promedio).

## 🔎 Laboratorio de profundización: función de pérdida y descenso por gradiente

La regresión lineal predice:

$$\hat y_i=w_0+\sum_{j=1}^{p}w_jx_{ij}$$

Una pérdida habitual es el error cuadrático medio:

$$MSE(w)=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2$$

El gradiente indica cómo cambia la pérdida. Descenso por gradiente actualiza:

$$w \leftarrow w-\eta\nabla_w MSE$$

`η` (`learning_rate`) y el número de iteraciones son hiperparámetros; los coeficientes `w` son parámetros aprendidos.


In [ ]:
# Paso 1: datos pequeños y pérdida explícita
import numpy as np

x_gd = np.array([0., 1., 2., 3., 4.])
y_gd = np.array([1., 3., 5., 7., 9.])
w, b = 0.0, 0.0

def mse(y_real, y_pred):
    return np.mean((y_real - y_pred) ** 2)

print("Pérdida inicial:", mse(y_gd, w * x_gd + b))


In [ ]:
# Paso 2: una actualización manual del gradiente
y_pred = w * x_gd + b
error = y_pred - y_gd
dw = (2 / len(x_gd)) * np.sum(error * x_gd)
db = (2 / len(x_gd)) * np.sum(error)
learning_rate = 0.05
w -= learning_rate * dw
b -= learning_rate * db
print({"w": w, "b": b, "mse": mse(y_gd, w * x_gd + b)})


In [ ]:
# Paso 3: repetir y observar convergencia
historial_perdida = []
for epoca in range(100):
    y_pred = w * x_gd + b
    error = y_pred - y_gd
    w -= learning_rate * (2 / len(x_gd)) * np.sum(error * x_gd)
    b -= learning_rate * (2 / len(x_gd)) * np.sum(error)
    historial_perdida.append(mse(y_gd, w * x_gd + b))

print({"pendiente": round(w, 3), "intercepto": round(b, 3),
       "mse_final": round(historial_perdida[-1], 6)})


### Regularización e hiperparámetros

Ridge añade $\alpha\sum w_j^2$ y Lasso añade $\alpha\sum |w_j|$ a la pérdida. `alpha` controla el compromiso entre ajustar los datos y mantener coeficientes pequeños. En polinomios, `degree` aumenta flexibilidad y riesgo de sobreajuste. Examina `coef_`, `intercept_`, `get_params()` y residuos después de `.fit()`.


<a id="aplicaciones"></a>
## 8. Ejemplos de Aplicación en el Mundo Real

- Predicción de precios de vivienda o de vehículos usados.
- Estimación del avance de una enfermedad para priorizar seguimiento médico (justo el caso de hoy).
- Pronóstico de ventas o de consumo energético.

<a id="retos"></a>
## 9. Retos de Práctica

### 🥉 Reto Básico
Entrena una Regresión Lineal simple usando solo la variable `s5` (una de las mediciones de sangre) y calcula MAE, RMSE y R² sobre el conjunto de prueba.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Entrena una Regresión Lineal múltiple con todas las variables y compara sus métricas contra el modelo del Reto Básico. ¿Mejoró el R²?

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Compara en una sola tabla Regresión Lineal, Ridge y Lasso probando al menos 3 valores distintos de `alpha` cada uno (ej. 0.1, 1.0, 10.0). Identifica con qué valor de `alpha` Lasso empieza a eliminar variables, y cuál combinación (modelo + alpha) da el mejor R² en prueba.

In [ ]:
# Tu solución al Reto Avanzado aquí
